# LeetCode 1968 — Array With Elements Not Equal to Average of Neighbors bằng Python

Link đề: https://leetcode.com/problems/array-with-elements-not-equal-to-average-of-neighbors/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Sorting

Skeleton LeetCode:

```python
class Solution:
    def rearrangeArray(self, nums: List[int]) -> List[int]:
        ...
```

Bài có **nhiều đáp án** — ta luôn kiểm bằng **hàm kiểm tra hợp lệ** thay vì so khớp một đáp án cố định. Notebook đi từ **thử các hoán vị** (vét cạn) → **sắp xếp rồi xen kẽ nửa nhỏ / nửa lớn** → **sắp xếp rồi đổi chỗ từng cặp kề** (dạng "răng cưa") → **răng cưa một lượt không cần sắp xếp** `O(n)`.

## 1. Mục tiêu học

1. Viết **hàm kiểm tra** đáp án cho bài nhiều lời giải: là hoán vị của `nums` và không có `nums[i]` bằng trung bình hai hàng xóm.
2. Hiểu ý tưởng then chốt: nếu mỗi phần tử ở giữa là **đỉnh** (lớn hơn cả hai hàng xóm) hoặc **đáy** (nhỏ hơn cả hai) thì nó **không thể** là trung bình của hai hàng xóm.
3. Cài đặt hai cách dựa trên sắp xếp (xen kẽ hai nửa; đổi cặp kề) và chứng minh chúng tạo mảng "răng cưa".
4. Cài đặt **wiggle sort** một lượt `O(n)` (LeetCode 280) và hiểu vì sao giá trị **khác nhau** là quan trọng.
5. Tránh lỗi kiểm tra trung bình bằng phép chia nguyên `//`.

## 2. Đề bài & ví dụ

Cho mảng `nums` gồm các số nguyên **đôi một khác nhau**. Hãy **sắp xếp lại** sao cho với mọi `1 ≤ i < n − 1`: `(nums[i−1] + nums[i+1]) / 2 ≠ nums[i]` — tức không phần tử nào bằng **trung bình cộng hai hàng xóm**. Trả về **một** cách sắp xếp bất kỳ thỏa mãn.

**Ví dụ 1:** `nums = [1,2,3,4,5]` → ví dụ `[1,2,4,5,3]`. Kiểm: `i=1`: `(1+4)/2 = 2.5 ≠ 2`; `i=2`: `(2+5)/2 = 3.5 ≠ 4`; `i=3`: `(4+3)/2 = 3.5 ≠ 5`.

**Ví dụ 2:** `nums = [6,2,0,9,7]` → ví dụ `[9,7,6,2,0]` (mảng gốc cũng đã thỏa).

**Ràng buộc:** `3 ≤ nums.length ≤ 10⁵`, `0 ≤ nums[i] ≤ 10⁵`, các giá trị khác nhau.

In [1]:
import random
import time
from collections import Counter
from itertools import permutations
from typing import List

examples = [[1, 2, 3, 4, 5], [6, 2, 0, 9, 7]]

def is_valid(original, arr):
    """Đáp án hợp lệ: là hoán vị của original và không phần tử nào bằng trung bình hai hàng xóm."""
    if Counter(original) != Counter(arr):
        return False
    return all(2 * arr[i] != arr[i - 1] + arr[i + 1] for i in range(1, len(arr) - 1))

print(is_valid([1, 2, 3, 4, 5], [1, 2, 4, 5, 3]), is_valid([6, 2, 0, 9, 7], [9, 7, 6, 2, 0]), is_valid([6, 2, 0, 9, 7], [6, 2, 0, 9, 7]))
print("mảng tăng dần [1..5] hợp lệ?", is_valid([1, 2, 3, 4, 5], [1, 2, 3, 4, 5]))

True True True
mảng tăng dần [1..5] hợp lệ? False


## 3. Kiến thức nền

### 3.1 So sánh "trung bình" không dùng số thực

`(a + c) / 2 == b` ⇔ `a + c == 2b`. Nhân chéo tránh hoàn toàn sai số và tránh lỗi chia nguyên.

In [2]:
a, b, c = 1, 2, 4
print((a + c) / 2 == b, a + c == 2 * b, "| sai nếu dùng //:", (a + c) // 2 == b)

False False | sai nếu dùng //: True


### 3.2 Đỉnh và đáy

`b` là **đỉnh** nếu `b > a` và `b > c`; **đáy** nếu `b < a` và `b < c`. Khi đó `2b > a + c` (đỉnh) hoặc `2b < a + c` (đáy) ⇒ `b` không phải trung bình.

In [3]:
for trip in [(1, 5, 2), (5, 1, 7), (1, 3, 5), (2, 2, 2)]:
    a, b, c = trip
    kind = "đỉnh" if b > a and b > c else ("đáy" if b < a and b < c else "không")
    print(trip, kind, "| là trung bình?", a + c == 2 * b)

(1, 5, 2) đỉnh | là trung bình? False
(5, 1, 7) đáy | là trung bình? False
(1, 3, 5) không | là trung bình? True
(2, 2, 2) không | là trung bình? True


### 3.3 Cắt lát và xen kẽ hai list

`zip(A, B)` ghép từng cặp; với hai list lệch độ dài 1, phần tử dư cần thêm riêng.

In [4]:
A, B = [1, 2, 3], [4, 5]
out = []
for x, y in zip(A, B):
    out += [x, y]
out += A[len(B):]
print(out)

[1, 4, 2, 5, 3]


### 3.4 `itertools.permutations`

Sinh mọi hoán vị (`n!`) — chỉ để vét cạn với `n` nhỏ.

In [5]:
print(sum(1 for _ in permutations(range(6))), "hoán vị của 6 phần tử")

720 hoán vị của 6 phần tử


## 4. Ý tưởng nền tảng / trực giác

Mảng **tăng dần** `[1,2,3,4,5]` là trường hợp tệ nhất: mọi phần tử giữa đều là trung bình (cấp số cộng). Cần "phá" tính đơn điệu.

Ý tưởng **răng cưa**: sắp xếp lại sao cho dãy lên–xuống luân phiên `a₀ < a₁ > a₂ < a₃ > …`. Mỗi phần tử ở giữa khi đó là đỉnh hoặc đáy ⇒ không là trung bình.

Tính tay với `[1,2,3,4,5]`: chia nửa nhỏ `[1,2,3]`, nửa lớn `[4,5]`, xen kẽ: `1, 4, 2, 5, 3` ⇒ `1 < 4 > 2 < 5 > 3` ✔. Hoặc sắp xếp rồi đổi chỗ cặp `(1,2)`, `(3,4)`: `1, 3, 2, 5, 4` ⇒ `1 < 3 > 2 < 5 > 4` ✔.

In [6]:
def pattern(arr):
    return ''.join('<' if arr[i] < arr[i + 1] else '>' for i in range(len(arr) - 1))

for arr in [[1, 2, 3, 4, 5], [1, 4, 2, 5, 3], [1, 3, 2, 5, 4], [9, 7, 6, 2, 0]]:
    print(arr, pattern(arr), "hợp lệ" if is_valid(arr, arr) else "không hợp lệ")

[1, 2, 3, 4, 5] <<<< không hợp lệ
[1, 4, 2, 5, 3] <><> hợp lệ
[1, 3, 2, 5, 4] <><> hợp lệ
[9, 7, 6, 2, 0] >>>> hợp lệ


Để ý `[9,7,6,2,0]` (đáp án đề) **không** răng cưa nhưng vẫn hợp lệ — răng cưa là điều kiện **đủ**, không cần thiết.

## 5. Cách 1 — Brute Force / baseline: thử hoán vị

In [7]:
def rearrange_brute(nums):
    for perm in permutations(nums):
        if all(2 * perm[i] != perm[i - 1] + perm[i + 1] for i in range(1, len(perm) - 1)):
            return list(perm)
    return None

for nums in examples:
    out = rearrange_brute(nums)
    print(nums, "->", out, is_valid(nums, out))

[1, 2, 3, 4, 5] -> [1, 2, 4, 3, 5] True
[6, 2, 0, 9, 7] -> [6, 2, 0, 9, 7] True


**Độ phức tạp:** xấu nhất `O(n! · n)`. Trên thực tế thường tìm thấy rất sớm (nhiều hoán vị hợp lệ), nhưng không có bảo đảm và không dùng được cho `n = 10⁵`.

**Nút thắt:** không có định hướng. Cần một **cấu trúc** bảo đảm hợp lệ — răng cưa.

## 6. Ý tưởng thuật toán / bước đột phá

**Quy tắc:** **nếu** mọi phần tử ở vị trí lẻ lớn hơn hai hàng xóm (hoặc mọi phần tử ở vị trí lẻ nhỏ hơn) **thì** không phần tử giữa nào là trung bình.

Ba cách tạo răng cưa:
1. **Xen kẽ hai nửa:** sắp tăng; nửa nhỏ `S = a[0..m−1]` (`m = ⌈n/2⌉`), nửa lớn `L = a[m..]`; đặt `S` vào vị trí chẵn, `L` vào vị trí lẻ. Mọi phần tử của `L` lớn hơn mọi phần tử của `S` ⇒ vị trí lẻ là đỉnh.
2. **Đổi cặp kề:** sắp tăng; đổi `a[1]↔a[2]`, `a[3]↔a[4]`, … ⇒ `a₀ < a₂ > a₁ < a₄ > a₃ …`.
3. **Wiggle một lượt:** duyệt `i = 1..n−1`; **nếu** `i` lẻ mà `a[i] < a[i−1]` **hoặc** `i` chẵn mà `a[i] > a[i−1]` **thì** đổi `a[i]↔a[i−1]`. Bất biến: tiền tố `a[0..i]` đã răng cưa.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp rồi xen kẽ nửa nhỏ / nửa lớn

In [8]:
def rearrange_halves(nums):
    a = sorted(nums)
    n = len(a)
    m = (n + 1) // 2                  # nửa nhỏ có ceil(n/2) phần tử
    out = []
    for i in range(m):
        out.append(a[i])              # vị trí chẵn: nửa nhỏ
        if i + m < n:
            out.append(a[i + m])      # vị trí lẻ: nửa lớn
    return out

for nums in examples:
    out = rearrange_halves(nums)
    print(nums, "->", out, pattern(out), is_valid(nums, out))

[1, 2, 3, 4, 5] -> [1, 4, 2, 5, 3] <><> True
[6, 2, 0, 9, 7] -> [0, 7, 2, 9, 6] <><> True


**Độ phức tạp:** `O(n log n)` thời gian (sắp xếp), `O(n)` bộ nhớ.

### 7.2 Cách 3 — Sắp xếp rồi đổi chỗ từng cặp kề

In [9]:
def rearrange_swap_pairs(nums):
    a = sorted(nums)
    for i in range(1, len(a) - 1, 2):
        a[i], a[i + 1] = a[i + 1], a[i]
    return a

for nums in examples:
    out = rearrange_swap_pairs(nums)
    print(nums, "->", out, pattern(out), is_valid(nums, out))

[1, 2, 3, 4, 5] -> [1, 3, 2, 5, 4] <><> True
[6, 2, 0, 9, 7] -> [0, 6, 2, 9, 7] <><> True


Sau khi đổi, vị trí lẻ `i` chứa `a_sorted[i+1]`, lớn hơn cả `a_sorted[i−1]`… (hàng xóm trái là phần tử nhỏ hơn) và `a_sorted[i]` (hàng xóm phải mới) ⇒ đỉnh. **Độ phức tạp:** `O(n log n)`, `O(n)` (hoặc `O(1)` nếu sắp tại chỗ).

### 7.3 Cách 4 — Wiggle một lượt, không sắp xếp (lời giải chính)

In [10]:
def rearrange_wiggle(nums):
    a = list(nums)
    for i in range(1, len(a)):
        if (i % 2 == 1 and a[i] < a[i - 1]) or (i % 2 == 0 and a[i] > a[i - 1]):
            a[i], a[i - 1] = a[i - 1], a[i]
    return a

for nums in examples:
    out = rearrange_wiggle(nums)
    print(nums, "->", out, pattern(out), is_valid(nums, out))

[1, 2, 3, 4, 5] -> [1, 3, 2, 5, 4] <><> True
[6, 2, 0, 9, 7] -> [2, 6, 0, 9, 7] <><> True


**Độ phức tạp:** `O(n)` thời gian, `O(n)` cho bản sao (có thể làm tại chỗ `O(1)`).

**Vì sao một lần đổi không phá phần trước?** Giả sử `i` lẻ và cần `a[i−1] < a[i]` nhưng đang có `a[i] < a[i−1]`. Trước đó (`i − 1` chẵn) ta đã có `a[i−2] > a[i−1]`. Đổi chỗ: vị trí `i−1` nhận `a[i]`, nhỏ hơn giá trị cũ ⇒ vẫn `< a[i−2]` ✔; vị trí `i` nhận `a[i−1]` cũ > `a[i]` ✔. Trường hợp `i` chẵn đối xứng.

### 7.4 Cách 5 — Ngẫu nhiên: xáo trộn tới khi hợp lệ (Las Vegas)

Vì phần lớn hoán vị đều hợp lệ (xem tỉ lệ dưới đây), một chiến lược đơn giản là **xáo trộn ngẫu nhiên** rồi kiểm tra, lặp lại nếu chưa đạt. Mỗi lần thử `O(n)`; số lần thử kỳ vọng nhỏ với mảng ngẫu nhiên — nhưng **không có bảo đảm** cận trên (một thuật toán Las Vegas: luôn đúng khi dừng, thời gian ngẫu nhiên). Không nên nộp khi đã có cách tất định `O(n)`.

In [11]:
def valid_fraction(nums):
    total = good = 0
    for p in permutations(nums):
        total += 1
        good += all(2 * p[i] != p[i - 1] + p[i + 1] for i in range(1, len(p) - 1))
    return good / total

for t in ([1, 2, 3, 4, 5], [0, 1, 2, 3, 4, 5, 6], [3, 1, 4, 15, 9, 2, 6]):
    print(f"{t}: tỉ lệ hoán vị hợp lệ = {valid_fraction(t):.3f}")

def rearrange_random(nums, seed=0):
    rnd_local = random.Random(seed)
    a = list(nums); tries = 0
    while True:
        tries += 1
        rnd_local.shuffle(a)
        if all(2 * a[i] != a[i - 1] + a[i + 1] for i in range(1, len(a) - 1)):
            return a, tries

big_ap = list(range(0, 3 * 2000, 3))                 # cấp số cộng 2000 phần tử
out, tries = rearrange_random(big_ap, seed=7)
print("cấp số cộng 2000 phần tử: hợp lệ =", is_valid(big_ap, out), "sau", tries, "lần xáo trộn")

[1, 2, 3, 4, 5]: tỉ lệ hoán vị hợp lệ = 0.667
[0, 1, 2, 3, 4, 5, 6]: tỉ lệ hoán vị hợp lệ = 0.651
[3, 1, 4, 15, 9, 2, 6]: tỉ lệ hoán vị hợp lệ = 0.781
cấp số cộng 2000 phần tử: hợp lệ = True sau 1 lần xáo trộn


## 8. TRACE từng bước

Trace cách 4 trên `[1, 2, 3, 4, 5]` (mục tiêu `a₀ < a₁ > a₂ < a₃ > a₄`):

| i | cần | trước | đổi? | sau |
|---|---|---|---|---|
| 1 | a₀ < a₁ | [1,2,3,4,5] | không | [1,2,3,4,5] |
| 2 | a₁ > a₂ | 2 vs 3 | đổi | [1,3,2,4,5] |
| 3 | a₂ < a₃ | 2 vs 4 | không | |
| 4 | a₃ > a₄ | 4 vs 5 | đổi | [1,3,2,5,4] |

In [12]:
def trace_wiggle(nums):
    a = list(nums)
    for i in range(1, len(a)):
        need = "<" if i % 2 == 1 else ">"
        bad = (i % 2 == 1 and a[i] < a[i - 1]) or (i % 2 == 0 and a[i] > a[i - 1])
        before = a[:]
        if bad:
            a[i], a[i - 1] = a[i - 1], a[i]
        print(f"  i={i}: cần a[{i - 1}] {need} a[{i}], trước {before}, {'đổi' if bad else 'giữ'} -> {a}")
    print("  mẫu:", pattern(a), "| hợp lệ:", is_valid(nums, a))
    return a

trace_wiggle([1, 2, 3, 4, 5])

  i=1: cần a[0] < a[1], trước [1, 2, 3, 4, 5], giữ -> [1, 2, 3, 4, 5]
  i=2: cần a[1] > a[2], trước [1, 2, 3, 4, 5], đổi -> [1, 3, 2, 4, 5]
  i=3: cần a[2] < a[3], trước [1, 3, 2, 4, 5], giữ -> [1, 3, 2, 4, 5]
  i=4: cần a[3] > a[4], trước [1, 3, 2, 4, 5], đổi -> [1, 3, 2, 5, 4]
  mẫu: <><> | hợp lệ: True


[1, 3, 2, 5, 4]

In [13]:
def trace_halves(nums):
    a = sorted(nums); n = len(a); m = (n + 1) // 2
    print(f"  sắp xếp: {a}; nửa nhỏ {a[:m]}, nửa lớn {a[m:]}")
    out = []
    for i in range(m):
        out.append(a[i])
        if i + m < n:
            out.append(a[i + m])
        print(f"  i={i}: thêm {a[i]}" + (f" và {a[i + m]}" if i + m < n else "") + f" -> {out}")
    return out

trace_halves([6, 2, 0, 9, 7])

  sắp xếp: [0, 2, 6, 7, 9]; nửa nhỏ [0, 2, 6], nửa lớn [7, 9]
  i=0: thêm 0 và 7 -> [0, 7]
  i=1: thêm 2 và 9 -> [0, 7, 2, 9]
  i=2: thêm 6 -> [0, 7, 2, 9, 6]


[0, 7, 2, 9, 6]

## 9. Code nộp LeetCode

In [14]:
class Solution:
    def rearrangeArray(self, nums: List[int]) -> List[int]:
        a = nums[:]
        for i in range(1, len(a)):
            if (i % 2 == 1 and a[i] < a[i - 1]) or (i % 2 == 0 and a[i] > a[i - 1]):
                a[i], a[i - 1] = a[i - 1], a[i]     # giữ răng cưa a0 < a1 > a2 < a3 ...
        return a

for nums in examples:
    out = Solution().rearrangeArray(nums)
    assert is_valid(nums, out)
print("Ví dụ đề: OK (kiểm bằng hàm hợp lệ)")

Ví dụ đề: OK (kiểm bằng hàm hợp lệ)


## 10. Vì sao thuật toán đúng

**Bổ đề:** nếu mảng có dạng răng cưa nghiêm ngặt (`a₀ < a₁ > a₂ < …` hoặc ngược lại) thì không phần tử giữa nào bằng trung bình hai hàng xóm. Vì mỗi `aᵢ` giữa là đỉnh (`2aᵢ > aᵢ₋₁ + aᵢ₊₁`) hoặc đáy (`2aᵢ < aᵢ₋₁ + aᵢ₊₁`).

**Cách 2 (xen kẽ hai nửa):** vị trí lẻ chứa phần tử của nửa lớn `L`, hai hàng xóm thuộc nửa nhỏ `S`; mọi phần tử `L` > mọi phần tử `S` (giá trị khác nhau, sắp tăng) ⇒ vị trí lẻ là đỉnh, vị trí chẵn là đáy ⇒ răng cưa.

**Cách 3 (đổi cặp):** với dãy tăng `b₀ < b₁ < …`, sau khi đổi được `b₀, b₂, b₁, b₄, b₃, …`; vị trí lẻ `2t+1` chứa `b_{2t+2}`, hai hàng xóm là `b_{2t−1}` hoặc `b₀` và `b_{2t+1}` — đều nhỏ hơn ⇒ đỉnh (phần tử cuối nếu dư thì chỉ có một hàng xóm, không cần xét).

**Cách 4 (wiggle):** bất biến "tiền tố `a[0..i]` răng cưa" được giữ (mục 7.3) ⇒ cuối cùng cả mảng răng cưa. Cần giá trị **khác nhau** để các so sánh nghiêm ngặt; nếu có trùng, răng cưa nghiêm ngặt có thể không tồn tại.

Cả ba cách đều trả về một **hoán vị** của `nums` (chỉ sắp xếp/đổi chỗ).

Kiểm chứng bằng máy: (1) mọi mảng răng cưa nghiêm ngặt đều hợp lệ; (2) điều ngược lại **không** đúng (có mảng hợp lệ không răng cưa, ví dụ đáp án đề `[9,7,6,2,0]`).

In [15]:
def is_zigzag(a):
    return all((a[i] - a[i - 1]) * (a[i + 1] - a[i]) < 0 for i in range(1, len(a) - 1))

rnd = random.Random(11)
zig = valid_not_zig = 0
for _ in range(3000):
    t = rnd.sample(range(0, 40), rnd.randint(3, 9))
    if is_zigzag(t):
        assert is_valid(t, t); zig += 1
    elif is_valid(t, t):
        valid_not_zig += 1
print(f"{zig} mảng răng cưa đều hợp lệ; {valid_not_zig} mảng hợp lệ nhưng không răng cưa")

726 mảng răng cưa đều hợp lệ; 2132 mảng hợp lệ nhưng không răng cưa


## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử hoán vị | `O(n! · n)` xấu nhất | `O(n)` |
| 2. Sắp xếp + xen kẽ | `O(n log n)` | `O(n)` |
| 3. Sắp xếp + đổi cặp | `O(n log n)` | `O(1)` phụ (tại chỗ) |
| 4. Wiggle một lượt | `O(n)` | `O(1)` phụ (tại chỗ) |

In [16]:
rnd = random.Random(1968)
big = rnd.sample(range(0, 10**5 + 1), 10**5)
for name, fn in [("halves", rearrange_halves), ("swap_pairs", rearrange_swap_pairs), ("wiggle", rearrange_wiggle), ("Solution", Solution().rearrangeArray)]:
    t0 = time.perf_counter(); out = fn(big); dt = time.perf_counter() - t0
    print(f"{name:<11}: hợp lệ = {is_valid(big, out)}, {dt * 1000:6.1f} ms")

halves     : hợp lệ = True,   18.7 ms


swap_pairs : hợp lệ = True,   18.0 ms


wiggle     : hợp lệ = True,   11.1 ms


Solution   : hợp lệ = True,   10.4 ms


## 12. Bộ test

Vì có nhiều đáp án, mọi test dùng `is_valid`. Thêm kiểm tra: input không bị sửa (nếu lời giải dùng bản sao), các mảng "khó" (tăng dần, giảm dần, cấp số cộng), và ngẫu nhiên có seed.

In [17]:
ALL = {
    "brute": rearrange_brute,
    "halves": rearrange_halves,
    "swap_pairs": rearrange_swap_pairs,
    "wiggle": rearrange_wiggle,
    "Solution": lambda a: Solution().rearrangeArray(a),
}

def check(nums, brute=True):
    snap = list(nums)
    for name, fn in ALL.items():
        if name == "brute" and (not brute or len(nums) > 8):
            continue
        out = fn(nums)
        assert out is not None and is_valid(nums, out), (name, nums, out)
        assert nums == snap, f"{name} đã sửa input"
    return True

for nums in examples:
    check(nums)
print("Ví dụ đề: OK")

for nums in ([1, 2, 3], [3, 2, 1], [0, 1, 2, 3, 4, 5, 6, 7], list(range(0, 100, 3)), list(range(50, 0, -1)), [0, 100000, 50000]):
    check(nums)
print("Edge case (tăng dần, giảm dần, cấp số cộng): OK")

rnd = random.Random(2005)
for _ in range(2000):
    n = rnd.randint(3, 8)
    check(rnd.sample(range(0, 30), n))
for _ in range(200):
    n = rnd.randint(3, 3000)
    check(rnd.sample(range(0, 10**5 + 1), n), brute=False)
print("2000 mảng nhỏ + 200 mảng cỡ lớn ngẫu nhiên: OK")

Ví dụ đề: OK
Edge case (tăng dần, giảm dần, cấp số cộng): OK


2000 mảng nhỏ + 200 mảng cỡ lớn ngẫu nhiên: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Kiểm tra trung bình bằng phép chia nguyên `//`

In [18]:
def is_valid_bug(original, arr):
    return all((arr[i - 1] + arr[i + 1]) // 2 != arr[i] for i in range(1, len(arr) - 1))   # SAI

arr = [1, 2, 4, 5, 3]                 # đáp án mẫu của đề — hợp lệ
print("kiểm bằng //:", is_valid_bug(arr, arr), "| kiểm đúng:", is_valid(arr, arr))

kiểm bằng //: False | kiểm đúng: True


**Vì sao sai:** `(1 + 4) // 2 = 2` nên hàm kiểm tra sai báo "bằng trung bình" dù trung bình thật là `2.5`. **Sửa:** so `a + c == 2b`.

### Lỗi 2 — Chỉ sắp xếp (hoặc trả nguyên mảng)

In [19]:
def rearrange_bug_sorted(nums):
    return sorted(nums)                         # SAI: dãy tăng có thể là cấp số cộng

nums = [1, 2, 3, 4, 5]
print("sắp xếp:", rearrange_bug_sorted(nums), "hợp lệ?", is_valid(nums, rearrange_bug_sorted(nums)))

sắp xếp: [1, 2, 3, 4, 5] hợp lệ? False


**Sửa:** tạo răng cưa (xen kẽ / đổi cặp / wiggle).

### Lỗi 3 — Xen kẽ hai nửa với `m = n // 2` khi `n` lẻ

In [20]:
def rearrange_bug_half_size(nums):
    a = sorted(nums); n = len(a); m = n // 2      # SAI khi n lẻ: nửa nhỏ thiếu một phần tử
    out = []
    for i in range(m):
        out += [a[i], a[i + m]]
    return out

nums = [1, 2, 3, 4, 5]
out = rearrange_bug_half_size(nums)
print("m = n//2:", out, "| là hoán vị?", sorted(out) == sorted(nums), "| hợp lệ?", is_valid(nums, out))

m = n//2: [1, 3, 2, 4] | là hoán vị? False | hợp lệ? False


**Vì sao sai:** với `n = 5`, vòng lặp chỉ thêm 4 phần tử, **mất** phần tử lớn nhất. Nếu sửa vội bằng cách thêm phần tử còn lại vào cuối, nửa "lớn" bắt đầu từ `a[2]` (phần tử giữa) và có thể tạo bộ ba hỏng. **Sửa:** `m = (n + 1) // 2` và chỉ thêm `a[i + m]` khi `i + m < n`.

### Lỗi 4 — Wiggle chỉ so sánh với một phía rồi đổi, không theo luân phiên chẵn/lẻ

In [21]:
def rearrange_bug_wiggle(nums):
    a = list(nums)
    for i in range(1, len(a) - 1):
        if 2 * a[i] == a[i - 1] + a[i + 1]:
            a[i], a[i + 1] = a[i + 1], a[i]         # SAI: sửa cục bộ có thể làm hỏng bộ ba trước
    return a

rnd = random.Random(4)
found = None
while found is None:
    t = rnd.sample(range(0, 12), rnd.randint(3, 7))
    if not is_valid(t, rearrange_bug_wiggle(t)):
        found = t
print("phản ví dụ:", found, "->", rearrange_bug_wiggle(found), "| đúng (wiggle):", Solution().rearrangeArray(found))

phản ví dụ: [1, 7, 8, 4, 2, 0, 10] -> [1, 7, 8, 4, 0, 2, 10] | đúng (wiggle): [1, 8, 4, 7, 0, 10, 2]


**Vì sao sai:** đổi `a[i]` với `a[i+1]` làm thay đổi bộ ba `(a[i−1], a[i], a[i+1])` **và** bộ ba kế tiếp, nhưng không kiểm lại bộ ba phía trước (`a[i−2], a[i−1], a[i]`) — có thể tạo vi phạm mới. **Sửa:** dùng bất biến răng cưa luân phiên (cách 4) — đã chứng minh không phá phần trước.

### Lỗi 5 — `return nums.sort()` (sắp xếp tại chỗ trả về `None`)

In [22]:
def rearrange_bug_none(nums):
    a = list(nums)
    return a.sort()                            # SAI: list.sort() trả về None

print("kết quả:", rearrange_bug_none([3, 1, 2]))
def rearrange_fixed(nums):
    a = sorted(nums)
    for i in range(1, len(a) - 1, 2):
        a[i], a[i + 1] = a[i + 1], a[i]
    return a
print("sửa:", rearrange_fixed([3, 1, 2]), is_valid([3, 1, 2], rearrange_fixed([3, 1, 2])))

kết quả: None
sửa: [1, 3, 2] True


**Vì sao sai:** `list.sort()` sắp **tại chỗ** và trả `None`; muốn một list mới thì dùng `sorted(...)`. Lỗi này khiến LeetCode báo kết quả `null`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài "trả về **một** cách sắp xếp thỏa điều kiện cục bộ giữa hàng xóm" ⇒ thử **răng cưa** (wiggle) — điều kiện mạnh, dễ dựng.
- "Không bằng trung bình hàng xóm" ⇐ "là đỉnh hoặc đáy".
- So sánh trung bình bằng **nhân chéo**.
- Bài nhiều đáp án: viết `is_valid` **trước** khi viết lời giải, và test bằng nó.
- Sắp xếp + xen kẽ hai nửa là mẹo kinh điển (Wiggle Sort II — LeetCode 324, khó hơn vì có trùng).

## 15. Biến thể của bài

1. **LeetCode 280 — Wiggle Sort:** `a₀ ≤ a₁ ≥ a₂ ≤ …` (cho phép bằng) — đúng cách 4.
2. **LeetCode 324 — Wiggle Sort II:** răng cưa **nghiêm ngặt** khi có **phần tử trùng** — cần trung vị + đặt chỉ số ảo.
3. **Không có ba phần tử liên tiếp tạo cấp số cộng** theo **bất kỳ** công sai: chính là bài này.
4. **Không có `i < j < k` (không cần liên tiếp) với `2a_j = a_i + a_k`** ("beautiful array", LeetCode 932): chia để trị theo lẻ/chẵn.
5. **Đếm số hoán vị hợp lệ:** quy hoạch động trên bitmask (n nhỏ).

In [23]:
def beautiful_array(n):
    """LeetCode 932: hoán vị 1..n không có i<j<k với 2*a[j] = a[i] + a[k]."""
    res = [1]
    while len(res) < n:
        res = [2 * x - 1 for x in res] + [2 * x for x in res]
    return [x for x in res if x <= n]

def no_3ap_any(arr):
    pos = {v: i for i, v in enumerate(arr)}
    n = len(arr)
    for i in range(n):
        for k in range(i + 2, n):
            s = arr[i] + arr[k]
            if s % 2 == 0 and s // 2 in pos and i < pos[s // 2] < k:
                return False
    return True

for n in (5, 8, 13):
    b = beautiful_array(n)
    print(n, b, no_3ap_any(b), "| cũng hợp lệ cho bài 1968:", is_valid(b, b))

5 [1, 5, 3, 2, 4] True | cũng hợp lệ cho bài 1968: True
8 [1, 5, 3, 7, 2, 6, 4, 8] True | cũng hợp lệ cho bài 1968: True
13 [1, 9, 5, 13, 3, 11, 7, 2, 10, 6, 4, 12, 8] True | cũng hợp lệ cho bài 1968: True


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử hoán vị | vét cạn | `O(n!·n)` | `O(n)` | kiểm thử `n ≤ 8` |
| 2. Xen kẽ hai nửa | nhỏ ở chẵn, lớn ở lẻ | `O(n log n)` | `O(n)` | dễ hiểu, dễ chứng minh |
| 3. Đổi cặp kề | sắp xếp + swap `(1,2),(3,4)…` | `O(n log n)` | `O(1)` | ngắn gọn |
| 4. Wiggle một lượt | sửa luân phiên | `O(n)` | `O(1)` | **nộp bài**, nhanh nhất |
| 5. Xáo trộn ngẫu nhiên | thử tới khi hợp lệ | `O(n)` kỳ vọng mỗi lần thử, không bảo đảm | `O(n)` | minh họa, khi cần đáp án "ngẫu nhiên" |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `rearrange_halves([10, 20, 30, 40, 50, 60])` và `rearrange_swap_pairs([10, 20, 30, 40, 50, 60])` trả về gì?

**Bài 2 (tính tay):** Trace cách 4 với `nums = [5, 4, 3, 2, 1, 0]`.

**Bài 3 (điền TODO):** Hoàn thiện `count_bad(arr)` đếm số chỉ số `i` vi phạm (bằng trung bình hàng xóm) — dùng để chấm điểm lời giải.

**Bài 4 (tự viết lại):** Viết lại cách 4 sao cho mảng có dạng **ngược** `a₀ > a₁ < a₂ > …` và kiểm hợp lệ.

**Bài 5 (mini challenge):** Đếm **số hoán vị hợp lệ** của `nums` (n ≤ 8) bằng DP bitmask `dp[mask][last2]` hoặc liệt kê có cắt nhánh; so với `permutations`.

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_bad(arr):
    # TODO: đếm i trong [1, n-2] với 2*arr[i] == arr[i-1] + arr[i+1]
    pass

# Bài 4 — TODO
def rearrange_wiggle_reverse(nums):
    # TODO: đảo điều kiện: i lẻ cần a[i] < a[i-1]
    pass

# Bài 5 — TODO
def count_valid_perms(nums):
    # TODO: DFS xây dần hoán vị, chỉ kiểm bộ ba cuối mỗi khi thêm phần tử
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Nửa nhỏ `[10,20,30]`, nửa lớn `[40,50,60]`.
- Mức 2: Xen kẽ: `10,40,20,50,30,60`.
- Mức 3: Đổi cặp: `10,30,20,50,40,60`.

**Bài 2**
- Mức 1: `i = 1` cần `a₀ < a₁`: `5 > 4` ⇒ đổi.
- Mức 2: Mảng thành `[4,5,3,2,1,0]`, `i = 2` cần `a₁ > a₂` ✔.
- Mức 3: Tiếp tục `i = 3, 4, 5`.

**Bài 3**
- Mức 1: Một dòng `sum(...)`.
- Mức 2: Nhân chéo.
- Mức 3: `count_bad = 0` ⇔ hợp lệ (khi đã là hoán vị).

**Bài 4**
- Mức 1: Đổi `<` thành `>` trong điều kiện.
- Mức 2: Bất biến tương tự.
- Mức 3: Kiểm `pattern(out)` bắt đầu bằng `>`.

**Bài 5**
- Mức 1: Khi thêm phần tử thứ `k ≥ 3`, chỉ bộ ba cuối mới thay đổi.
- Mức 2: DFS với `used`.
- Mức 3: So với `sum(1 for p in permutations(nums) if valid)`.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1
r1 = (rearrange_halves([10, 20, 30, 40, 50, 60]), rearrange_swap_pairs([10, 20, 30, 40, 50, 60]))
print("Bài 1:", r1)
assert r1 == ([10, 40, 20, 50, 30, 60], [10, 30, 20, 50, 40, 60])

# Bài 2
out2 = trace_wiggle([5, 4, 3, 2, 1, 0])
assert is_valid([5, 4, 3, 2, 1, 0], out2)

# Bài 3
def count_bad_sol(arr):
    return sum(2 * arr[i] == arr[i - 1] + arr[i + 1] for i in range(1, len(arr) - 1))

assert count_bad_sol([1, 2, 3, 4, 5]) == 3 and count_bad_sol([1, 2, 4, 5, 3]) == 0
print("Bài 3 OK:", count_bad_sol([1, 2, 3, 4, 5]), count_bad_sol([1, 2, 4, 5, 3]))

Bài 1: ([10, 40, 20, 50, 30, 60], [10, 30, 20, 50, 40, 60])
  i=1: cần a[0] < a[1], trước [5, 4, 3, 2, 1, 0], đổi -> [4, 5, 3, 2, 1, 0]
  i=2: cần a[1] > a[2], trước [4, 5, 3, 2, 1, 0], giữ -> [4, 5, 3, 2, 1, 0]
  i=3: cần a[2] < a[3], trước [4, 5, 3, 2, 1, 0], đổi -> [4, 5, 2, 3, 1, 0]
  i=4: cần a[3] > a[4], trước [4, 5, 2, 3, 1, 0], giữ -> [4, 5, 2, 3, 1, 0]
  i=5: cần a[4] < a[5], trước [4, 5, 2, 3, 1, 0], đổi -> [4, 5, 2, 3, 0, 1]
  mẫu: <><>< | hợp lệ: True
Bài 3 OK: 3 0


In [26]:
# Bài 4
def rearrange_wiggle_reverse_sol(nums):
    a = list(nums)
    for i in range(1, len(a)):
        if (i % 2 == 1 and a[i] > a[i - 1]) or (i % 2 == 0 and a[i] < a[i - 1]):
            a[i], a[i - 1] = a[i - 1], a[i]
    return a

rnd = random.Random(4)
for _ in range(1000):
    t = rnd.sample(range(0, 1000), rnd.randint(3, 60))
    out = rearrange_wiggle_reverse_sol(t)
    assert is_valid(t, out) and pattern(out)[0] == '>'
print("Bài 4 OK | ví dụ:", rearrange_wiggle_reverse_sol([1, 2, 3, 4, 5]))

# Bài 5
def count_valid_perms_sol(nums):
    n = len(nums); used = [False] * n; cur = []; total = 0
    def dfs():
        nonlocal total
        if len(cur) == n:
            total += 1; return
        for i in range(n):
            if not used[i]:
                if len(cur) >= 2 and 2 * cur[-1] == cur[-2] + nums[i]:
                    continue
                used[i] = True; cur.append(nums[i]); dfs(); cur.pop(); used[i] = False
    dfs()
    return total

rnd = random.Random(5)
for _ in range(40):
    t = rnd.sample(range(0, 15), rnd.randint(3, 7))
    brute = sum(1 for p in permutations(t) if all(2 * p[i] != p[i - 1] + p[i + 1] for i in range(1, len(p) - 1)))
    assert count_valid_perms_sol(t) == brute
print("Bài 5 OK | [1,2,3,4,5] có", count_valid_perms_sol([1, 2, 3, 4, 5]), "hoán vị hợp lệ trên 120")

Bài 4 OK | ví dụ: [2, 1, 4, 3, 5]
Bài 5 OK | [1,2,3,4,5] có 80 hoán vị hợp lệ trên 120


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template wiggle một lượt (`a₀ < a₁ > a₂ < …`):**

```python
a = nums[:]
for i in range(1, len(a)):
    if (i % 2 == 1) == (a[i] < a[i - 1]):     # sai chiều mong muốn
        a[i], a[i - 1] = a[i - 1], a[i]
return a
```

**Template xen kẽ hai nửa:** `a.sort(); m = (n + 1) // 2; out = a[0], a[m], a[1], a[m+1], …`.

**Checklist:**
- [ ] Kết quả là **hoán vị** của `nums`?
- [ ] Kiểm trung bình bằng `a + c == 2b` (không dùng `//`)?
- [ ] Xen kẽ: `m = (n + 1) // 2` cho nửa nhỏ?
- [ ] Wiggle: điều kiện luân phiên theo chẵn/lẻ của `i`?
- [ ] Đã test bằng `is_valid` (không so khớp một đáp án cố định)?
- [ ] Chữ ký đúng: `def rearrangeArray(self, nums: List[int]) -> List[int]`?